# 1. Audio Download & Speech Classification

Fetches recording rows from the Postgres database, downloads each recording's audio into `data/audio/` (gitignored), runs Silero VAD to determine whether the recording contains speech, and writes a CSV to `data/audio_speech_labels.csv` containing the original DB columns plus a single extra `is_speech` (boolean) column.

Rows that fail to download or classify are skipped and logged to stdout — they are not written to the CSV, so the output stays exactly "DB row + is_speech".

Re-running the notebook resumes: rows whose id is already present in the CSV are skipped, and audio already downloaded to `data/audio/` is not re-fetched.

In [1]:
import os
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
import torch
from dotenv import load_dotenv
from scipy.io import wavfile
from silero_vad import get_speech_timestamps, load_silero_vad
from tqdm.notebook import tqdm

load_dotenv()

True

## Configuration

All values come from `.env` at the repo root (found automatically by `load_dotenv()` walking up from this notebook's directory), with sensible fallbacks.

In [2]:
# Database
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
TABLE_NAME = os.getenv("TABLE_NAME")

# Name of the column containing the streamable audio URL.
AUDIO_URL_COLUMN = os.getenv("AUDIO_URL_COLUMN", "streamableUrl")

# Name of the row identifier column, used for resuming and for naming
# downloaded audio files on disk.
ID_COLUMN = os.getenv("ID_COLUMN", "id")

_where_clause_env = os.getenv("WHERE_CLAUSE", "").strip()
WHERE_CLAUSE = _where_clause_env if _where_clause_env else None

_limit_env = os.getenv("LIMIT", "").strip()
LIMIT = int(_limit_env) if _limit_env else None

# Output locations (relative to this notebook's directory, i.e. jupyter_notebooks/)
DATA_DIR = Path(os.getenv("DATA_DIR", "../data"))
AUDIO_DIR = DATA_DIR / "audio"
OUTPUT_CSV = DATA_DIR / os.getenv("OUTPUT_CSV", "audio_speech_labels.csv")
SAVE_EVERY = int(os.getenv("SAVE_EVERY", "10"))

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Speech detection
SAMPLE_RATE = int(os.getenv("SAMPLE_RATE", "16000"))

# A recording is labelled as speech when at least this proportion of it
# contains detected speech.
SPEECH_RATIO_THRESHOLD = float(os.getenv("SPEECH_RATIO_THRESHOLD", "0.20"))

MIN_SPEECH_DURATION_MS = int(os.getenv("MIN_SPEECH_DURATION_MS", "250"))
MIN_SILENCE_DURATION_MS = int(os.getenv("MIN_SILENCE_DURATION_MS", "300"))
SPEECH_PAD_MS = int(os.getenv("SPEECH_PAD_MS", "100"))
DOWNLOAD_TIMEOUT_SECONDS = int(os.getenv("DOWNLOAD_TIMEOUT_SECONDS", "180"))

# Number of parallel workers. Each gets its own Silero model instance.
MAX_WORKERS = int(os.getenv("MAX_WORKERS", "8"))

torch.set_num_threads(1)

## Database

In [3]:
def connect_to_database():
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )


def load_database_rows() -> pd.DataFrame:
    query = f"SELECT * FROM {TABLE_NAME}"

    if WHERE_CLAUSE:
        query += f" WHERE {WHERE_CLAUSE}"

    if LIMIT is not None:
        query += f" LIMIT {int(LIMIT)}"

    print("Reading rows from PostgreSQL...")

    with connect_to_database() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            column_names = [description.name for description in cursor.description]
            rows = cursor.fetchall()

    dataframe = pd.DataFrame(rows, columns=column_names)

    print(f"Loaded {len(dataframe)} rows.")

    return dataframe

## Audio download (persisted to `data/audio/`)

In [4]:
def download_audio(
    url: str,
    destination_path: Path,
    sample_rate: int = SAMPLE_RATE,
    timeout_seconds: int = DOWNLOAD_TIMEOUT_SECONDS,
) -> None:
    """Download and decode `url` into a mono WAV file at `destination_path` via ffmpeg."""
    command = [
        "ffmpeg",
        "-y",
        "-loglevel", "error",
        "-rw_timeout", str(timeout_seconds * 1_000_000),
        "-i", str(url),
        "-vn",
        "-ac", "1",
        "-ar", str(sample_rate),
        str(destination_path),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=timeout_seconds,
    )

    if result.returncode != 0:
        error_message = result.stderr.decode("utf-8", errors="ignore").strip()
        raise RuntimeError(error_message or "ffmpeg failed to decode the audio.")

    if not destination_path.exists() or destination_path.stat().st_size == 0:
        raise ValueError("Downloaded audio file is empty.")

## Silero VAD classification

In [5]:
_thread_local = threading.local()


def get_vad_model():
    if not hasattr(_thread_local, "model"):
        _thread_local.model = load_silero_vad()
    return _thread_local.model


def classify_audio(audio_path: Path) -> bool:
    """Return True if the audio at `audio_path` is majority speech."""
    sample_rate, samples = wavfile.read(str(audio_path))

    if samples.ndim > 1:
        samples = samples.mean(axis=1)

    if np.issubdtype(samples.dtype, np.integer):
        max_value = float(np.iinfo(samples.dtype).max)
        samples = samples.astype(np.float32) / max_value
    else:
        samples = samples.astype(np.float32)

    waveform = torch.from_numpy(samples)

    total_duration_seconds = waveform.numel() / sample_rate

    speech_segments = get_speech_timestamps(
        waveform,
        get_vad_model(),
        sampling_rate=sample_rate,
        return_seconds=True,
        min_speech_duration_ms=MIN_SPEECH_DURATION_MS,
        min_silence_duration_ms=MIN_SILENCE_DURATION_MS,
        speech_pad_ms=SPEECH_PAD_MS,
    )

    speech_duration_seconds = sum(
        max(0.0, float(segment["end"]) - float(segment["start"]))
        for segment in speech_segments
    )

    if total_duration_seconds > 0:
        speech_ratio = speech_duration_seconds / total_duration_seconds
    else:
        speech_ratio = 0.0

    return speech_ratio >= SPEECH_RATIO_THRESHOLD

## Row worker

Downloads (if not already on disk), classifies, and returns the DB row with an `is_speech` column appended. Returns `None` on failure so the row is skipped rather than written with bad data.

In [6]:
def process_row(row: dict):
    row_id = row.get(ID_COLUMN)
    audio_url = row.get(AUDIO_URL_COLUMN)

    try:
        if pd.isna(audio_url) or not str(audio_url).strip():
            raise ValueError("Audio URL is missing.")

        audio_path = AUDIO_DIR / f"{row_id}.wav"

        if not audio_path.exists():
            download_audio(str(audio_url).strip(), audio_path)

        result = dict(row)
        result["is_speech"] = classify_audio(audio_path)
        return result

    except Exception as error:
        print(f"[FAILED] {ID_COLUMN}={row_id}: {error}")
        return None

## Resume support

In [7]:
def load_existing_results() -> pd.DataFrame:
    if not OUTPUT_CSV.exists():
        return pd.DataFrame()

    try:
        return pd.read_csv(OUTPUT_CSV)
    except Exception as error:
        print(f"Could not read existing CSV: {error}")
        return pd.DataFrame()


def get_processed_ids(existing_results: pd.DataFrame) -> set:
    if existing_results.empty or ID_COLUMN not in existing_results.columns:
        return set()
    return set(existing_results[ID_COLUMN].astype(str).tolist())


def save_results(existing_results: pd.DataFrame, new_results: list) -> pd.DataFrame:
    new_dataframe = pd.DataFrame(new_results)

    if existing_results.empty:
        combined_dataframe = new_dataframe
    else:
        combined_dataframe = pd.concat([existing_results, new_dataframe], ignore_index=True)

    combined_dataframe.to_csv(OUTPUT_CSV, index=False)
    return combined_dataframe

## Run pipeline

In [ ]:
dataframe = load_database_rows()

if dataframe.empty:
    print("No database rows found.")
else:
    if AUDIO_URL_COLUMN not in dataframe.columns:
        raise ValueError(
            f"Column '{AUDIO_URL_COLUMN}' was not found.\n"
            f"Available columns: {list(dataframe.columns)}"
        )

    if ID_COLUMN not in dataframe.columns:
        raise ValueError(
            f"Column '{ID_COLUMN}' was not found.\n"
            f"Available columns: {list(dataframe.columns)}"
        )

    existing_results = load_existing_results()
    processed_ids = get_processed_ids(existing_results)

    rows_to_process = dataframe[~dataframe[ID_COLUMN].astype(str).isin(processed_ids)]

    print(f"Already processed: {len(processed_ids)}")
    print(f"Remaining rows:    {len(rows_to_process)}")
    print(f"Workers:           {MAX_WORKERS}")

    new_results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(process_row, row.to_dict())
            for _, row in rows_to_process.iterrows()
        ]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing audio", unit="audio"):
            result = future.result()

            if result is not None:
                new_results.append(result)

            if new_results and len(new_results) % SAVE_EVERY == 0:
                existing_results = save_results(existing_results, new_results)
                new_results = []

    if new_results:
        existing_results = save_results(existing_results, new_results)

    print("\nFinished processing.")
    print(f"Results saved to: {OUTPUT_CSV}")

Reading rows from PostgreSQL...
Loaded 6461 rows.
Already processed: 0
Remaining rows:    6461
Workers:           8
[FAILED] id=1a79d82d-1c18-4549-a157-4bd6be7fc390: [in#0 @ 0x8f300c000] Error opening input: Server returned 403 Forbidden (access denied)
Error opening input file https://says-api-streamable-audio-dev.s3.amazonaws.com/saysuseraudio1612892981787.mp3.
Error opening input files: Server returned 403 Forbidden (access denied)
[FAILED] id=4c29a182-ed6b-413a-aaf1-d3c0a7780ad6: [in#0 @ 0xcaf408000] Error opening input: Server returned 403 Forbidden (access denied)
Error opening input file https://says-api-streamable-audio-dev.s3.amazonaws.com/saysuseraudio1612889418870.mp3.
Error opening input files: Server returned 403 Forbidden (access denied)
[FAILED] id=3c940314-9e9d-4cef-b682-86a2a5d39af9: Audio URL is missing.
[FAILED] id=3f0e3e11-cf89-4255-a03e-267a972fabd7: Audio URL is missing.
[FAILED] id=fb0138c1-9f3f-4faf-9643-960706fe6f96: Audio URL is missing.
[FAILED] id=0e4abb65-e